# CTI Dataset Builder — Authenticated Hugging Face Download

This notebook downloads and inspects:

1. `threatcluster/cve-exploitation-signals`
2. `CIRCL/vulnerability-attack-techniques`

It authenticates with Hugging Face before loading the datasets, checks schemas and missing values, standardizes CVE IDs, checks cross-source overlap, builds a **row-wise union** that preserves source-specific fields, and exports CSV/Parquet files.

**Important:** Login does not fill values that are genuinely missing in the source. This notebook reports missingness rather than inventing data. Both datasets may be publicly accessible without authentication; logging in can help with account access/rate limits, but does not guarantee more complete records.

**Security:** Never paste your Hugging Face token into a notebook cell, Git repository, or shared output. Use the login prompt and keep the token private.

## 1. Install required packages

## 2. Log in to Hugging Face
Run this cell and follow the private token prompt. Create a read token in your Hugging Face account settings if needed.

In [2]:
from huggingface_hub import login, whoami

# Prompts securely for a token; do not hard-code your token in this notebook.
login()

try:
    print("Authenticated Hugging Face account:", whoami()["name"])
except Exception as exc:
    print("Login may have completed, but account verification failed:", repr(exc))

Authenticated Hugging Face account: sachinchakrawarti


## 3. Configure dataset IDs and output paths

In [3]:
from datasets import load_dataset
from huggingface_hub import HfApi
import pandas as pd
import numpy as np
import re
from pathlib import Path

DATASET_IDS = {
    "threatcluster": "threatcluster/cve-exploitation-signals",
    "circl": "CIRCL/vulnerability-attack-techniques",
}

OUTPUT_DIR = Path("cti_dataset_output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

api = HfApi()
print("Output directory:", OUTPUT_DIR.resolve())

Output directory: D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new\cti_dataset_output


## 4. Check access and load each dataset
This loads the default configuration/split exposed by `load_dataset`. If a repository requires approval or has multiple configurations, the error message will help identify the next step.

In [4]:
loaded = {}
load_errors = {}

for source, dataset_id in DATASET_IDS.items():
    print(f"\\n{'=' * 80}\\nLoading {dataset_id}")
    try:
        info = api.dataset_info(dataset_id)
        print("Repository access confirmed:", info.id)
        ds_dict = load_dataset(dataset_id, token=True)
        loaded[source] = ds_dict
        print("Splits:", list(ds_dict.keys()))
        for split_name, split_data in ds_dict.items():
            print(f"  {split_name}: {len(split_data):,} rows; columns={split_data.column_names}")
    except Exception as exc:
        load_errors[source] = repr(exc)
        print("FAILED:", repr(exc))

if load_errors:
    print("\\nOne or more datasets could not be loaded. Review the error above and confirm your account has access.")
if not loaded:
    raise RuntimeError("No datasets loaded. Resolve the Hugging Face access error before continuing.")

\n================================================================================\nLoading threatcluster/cve-exploitation-signals
Repository access confirmed: threatcluster/cve-exploitation-signals
Splits: ['train']
  train: 60,879 rows; columns=['cve_id', 'description', 'published_date', 'cvss_v3_score', 'cvss_v3_severity', 'cvss_v3_vector', 'cwe_ids', 'affected_vendors', 'affected_products', 'in_kev', 'kev_added_date', 'kev_due_date', 'ransomware_use', 'has_exploit', 'exploit_count']
\n================================================================================\nLoading CIRCL/vulnerability-attack-techniques
Repository access confirmed: CIRCL/vulnerability-attack-techniques
Splits: ['train', 'test']
  train: 1,086 rows; columns=['id', 'title', 'description', 'exploitation_techniques', 'primary_impact', 'secondary_impact', 'techniques', 'techniques_derived', 'label_sources', 'attack_version', 'cvss_vector', 'cvss_version', 'cwes', 'affected_products', 'cpes', 'cwes_predicted']
  t

## 5. Select the default split and inspect schemas
For repositories with multiple splits, this uses `train` if available; otherwise it uses the first split.

In [5]:
raw_frames = {}

for source, ds_dict in loaded.items():
    split_name = "train" if "train" in ds_dict else next(iter(ds_dict.keys()))
    df = ds_dict[split_name].to_pandas()
    df["source"] = source
    raw_frames[source] = df
    print(f"\\n### {source} | split={split_name} | rows={len(df):,}")
    display(pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "non_null": [int(df[c].notna().sum()) for c in df.columns],
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "missing_pct": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    }))

\n### threatcluster | split=train | rows=60,879


,column,dtype,non_null,missing,missing_pct
0,cve_id,object,60879,0,0.00
1,description,object,60879,0,0.00
2,published_date,datetime64[s],60879,0,0.00
3,cvss_v3_score,object,54328,6551,10.76
4,cvss_v3_severity,object,54328,6551,10.76
5,cvss_v3_vector,object,54328,6551,10.76
6,cwe_ids,object,55468,5411,8.89
7,affected_vendors,object,41651,19228,31.58
8,affected_products,object,41654,19225,31.58
9,in_kev,bool,60879,0,0.00


\n### circl | split=train | rows=1,086


,column,dtype,non_null,missing,missing_pct
0,id,object,1086,0,0.0
1,title,object,1086,0,0.0
2,description,object,1086,0,0.0
3,exploitation_techniques,object,1086,0,0.0
4,primary_impact,object,1086,0,0.0
5,secondary_impact,object,1086,0,0.0
6,techniques,object,1086,0,0.0
7,techniques_derived,object,1086,0,0.0
8,label_sources,object,1086,0,0.0
9,attack_version,object,1086,0,0.0


## 6. Clean column names and normalize CVE IDs
This removes accidental Markdown/HTML artifacts from column names (if present), without modifying cell contents beyond trimming whitespace and standardizing the CVE identifier.

In [6]:
def clean_column_name(name):
    name = str(name).strip()
    # Decode common HTML entities and remove accidental markdown emphasis.
    name = (name.replace("&#x61;", "a")
                .replace("&#x74;", "t")
                .replace("&amp;", "&"))
    name = name.replace("**", "").replace("*", "")
    name = re.sub(r"\\s+", "_", name)
    name = re.sub(r"[^0-9A-Za-z_]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name.lower()

def normalize_cve(value):
    if pd.isna(value):
        return pd.NA
    value = str(value).strip().upper()
    match = re.search(r"CVE-\\d{4}-\\d{4,}", value)
    return match.group(0) if match else pd.NA

clean_frames = {}
for source, original in raw_frames.items():
    df = original.copy()
    df.columns = [clean_column_name(c) for c in df.columns]
    # If cleanup creates duplicate column names, keep the first and report it.
    duplicated = df.columns[df.columns.duplicated()].tolist()
    if duplicated:
        print(f"{source}: duplicate column names after cleanup: {duplicated}; keeping first occurrence.")
        df = df.loc[:, ~df.columns.duplicated()]
    if "cve_id" not in df.columns:
        # CIRCL commonly calls its identifier `id`.
        if "id" in df.columns:
            df = df.rename(columns={"id": "cve_id"})
        else:
            raise KeyError(f"{source} dataset has no recognizable CVE identifier column.")
    df["cve_id"] = df["cve_id"].map(normalize_cve).astype("string")
    df["source"] = source
    clean_frames[source] = df

for source, df in clean_frames.items():
    print(source, "rows:", len(df), "valid CVE IDs:", int(df["cve_id"].notna().sum()))
    display(df.head(3))

threatcluster rows: 60879 valid CVE IDs: 0


,cve_id,description,published_date,cvss_v3_score,cvss_v3_severity,cvss_v3_vector,cwe_ids,affected_vendors,affected_products,in_kev,kev_added_date,kev_due_date,ransomware_use,has_exploit,exploit_count,source
0,<NA>,A vulnerability was detected in Tenda CP3 27.5...,2026-09-06,9.1,CRITICAL,CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H,"[CWE-77, CWE-78]",None,None,False,NaT,NaT,None,False,0,threatcluster
1,<NA>,A flaw has been found in code-projects Daily E...,2026-09-06,5.3,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:N/A:N,"[CWE-200, CWE-284]",None,None,False,NaT,NaT,None,False,0,threatcluster
2,<NA>,A security flaw has been discovered in itsourc...,2026-09-06,6.3,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:L/UI:N/S:U/C:L/I:L/A:L,"[CWE-74, CWE-89]",None,None,False,NaT,NaT,None,False,0,threatcluster


circl rows: 1086 valid CVE IDs: 0


,cve_id,title,description,exploitation_techniques,primary_impact,secondary_impact,techniques,techniques_derived,label_sources,attack_version,cvss_vector,cvss_version,cwes,affected_products,cpes,cwes_predicted,source
0,<NA>,Junos OS: SRX Series: A vulnerability in J-Web...,A Missing Authentication for Critical Function...,[T1190],[T1059],[],"[T1059, T1190]","[T1014, T1027.009, T1037, T1040, T1080, T1134,...",[ctid_kev],19.1,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:L/A:N,3.1,[CWE-306 Missing Authentication for Critical F...,[Juniper Networks Junos OS],"[cpe:2.3:h:juniper:ex2200:-:*:*:*:*:*:*:*, cpe...",[CWE-312 Cleartext Storage of Sensitive Inform...,circl
1,<NA>,,An elevation of privilege exists in Windows Au...,[],[T1068],[],[T1068],[],[ctid_cve],19.1,,,[Elevation of Privilege],"[Microsoft Windows, Microsoft Windows Server, ...",[cpe:2.3:o:microsoft:windows_10:-:*:*:*:*:*:*:...,[CWE-862 Missing Authorization],circl
2,<NA>,Cisco Webex Network Recording Player and Cisco...,A vulnerability in Cisco Webex Network Recordi...,"[T1204.002, T1566]",[T1499.004],[],"[T1204.002, T1499.004, T1566]","[T1027, T1036.001, T1539, T1553.002, T1562.003...",[ctid_cve],19.1,CVSS:3.1/AV:L/AC:L/PR:N/UI:R/S:U/C:N/I:N/A:L,3.1,[CWE-20 Improper Input Validation],"[Cisco Cisco Webex Network Recording Player, C...",[cpe:2.3:a:cisco:webex_network_recording_playe...,[CWE-404 Improper Resource Shutdown or Release],circl


## 7. Compare missingness and detect duplicate CVE IDs

In [7]:
for source, df in clean_frames.items():
    print(f"\\n### {source}")
    print("Duplicate CVE rows:", int(df["cve_id"].dropna().duplicated().sum()))
    missing_report = pd.DataFrame({
        "column": df.columns,
        "missing": [int(df[c].isna().sum()) for c in df.columns],
        "missing_pct": [round(float(df[c].isna().mean() * 100), 2) for c in df.columns],
    }).sort_values(["missing_pct", "column"], ascending=[False, True])
    display(missing_report)

cve_sets = {
    source: set(df["cve_id"].dropna().astype(str))
    for source, df in clean_frames.items()
}
sources_present = list(cve_sets)
if len(sources_present) >= 2:
    left, right = sources_present[:2]
    overlap = cve_sets[left] & cve_sets[right]
    print(f"\nCVE IDs in both {left} and {right}: {len(overlap):,}")
    print("Sample overlapping IDs:", sorted(overlap)[:20])

\n### threatcluster
Duplicate CVE rows: 0


,column,missing,missing_pct
0,cve_id,60879,100.00
10,kev_added_date,60759,99.80
11,kev_due_date,60759,99.80
12,ransomware_use,60759,99.80
8,affected_products,19225,31.58
7,affected_vendors,19228,31.58
3,cvss_v3_score,6551,10.76
4,cvss_v3_severity,6551,10.76
5,cvss_v3_vector,6551,10.76
6,cwe_ids,5411,8.89


\n### circl
Duplicate CVE rows: 0


,column,missing,missing_pct
0,cve_id,1086,100.0
13,affected_products,0,0.0
9,attack_version,0,0.0
14,cpes,0,0.0
10,cvss_vector,0,0.0
11,cvss_version,0,0.0
12,cwes,0,0.0
15,cwes_predicted,0,0.0
2,description,0,0.0
3,exploitation_techniques,0,0.0



CVE IDs in both threatcluster and circl: 0
Sample overlapping IDs: []


## 8. Build a row-wise union without discarding source-specific fields

Each source record remains a separate row. `uid` is the unique row key; `cve_id` is the cross-source matching key and can repeat. Source fields are retained as columns; fields unavailable in one source remain missing.

No missing values are filled with fabricated values. `threat_category` is initialized as blank unless a source explicitly provides a `threat_category` field. CVE datasets alone do not reliably label broader threat families such as phishing, malware, or botnets.

In [8]:
# Concatenate by column name, preserving fields from both sources.
combined = pd.concat(list(clean_frames.values()), ignore_index=True, sort=False)

# Create a unique row ID based on source + row position, independent of whether CVE ID exists.
combined.insert(
    0, "uid",
    [f"{src}_{i:07d}" for src, i in zip(combined["source"].astype(str),
                                        combined.groupby("source", sort=False).cumcount())]
)

if "threat_category" not in combined.columns:
    combined["threat_category"] = pd.NA

# Preserve dates separately; do not substitute the CVE year for publication date.
for date_col in ["published_date", "kev_added_date", "kev_due_date"]:
    if date_col in combined.columns:
        combined[date_col] = pd.to_datetime(combined[date_col], errors="coerce").dt.date

print("Combined rows:", f"{len(combined):,}")
print("Unique uid:", combined["uid"].nunique() == len(combined))
print("Unique non-null CVE IDs:", combined["cve_id"].nunique(dropna=True))
display(combined.head(10))

Combined rows: 61,965
Unique uid: True
Unique non-null CVE IDs: 0


,uid,cve_id,description,published_date,cvss_v3_score,cvss_v3_severity,cvss_v3_vector,cwe_ids,affected_vendors,affected_products,...,techniques,techniques_derived,label_sources,attack_version,cvss_vector,cvss_version,cwes,cpes,cwes_predicted,threat_category
0,threatcluster_0000000,<NA>,A vulnerability was detected in Tenda CP3 27.5...,2026-09-06,9.1,CRITICAL,CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H,"[CWE-77, CWE-78]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
1,threatcluster_0000001,<NA>,A flaw has been found in code-projects Daily E...,2026-09-06,5.3,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:N/A:N,"[CWE-200, CWE-284]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
2,threatcluster_0000002,<NA>,A security flaw has been discovered in itsourc...,2026-09-06,6.3,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:L/UI:N/S:U/C:L/I:L/A:L,"[CWE-74, CWE-89]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
3,threatcluster_0000003,<NA>,A vulnerability was determined in Tenda HG10 3...,2026-09-06,8.8,HIGH,CVSS:3.1/AV:N/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H,"[CWE-119, CWE-120]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
4,threatcluster_0000004,<NA>,A vulnerability was identified in itsourcecode...,2026-09-06,6.3,MEDIUM,CVSS:3.1/AV:N/AC:L/PR:L/UI:N/S:U/C:L/I:L/A:L,"[CWE-74, CWE-89]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
5,threatcluster_0000005,<NA>,A vulnerability has been found in code-project...,2026-09-06,7.3,HIGH,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:L/I:L/A:L,"[CWE-74, CWE-89]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
6,threatcluster_0000006,<NA>,A vulnerability has been found in Tenda CP3 27...,2026-09-06,9.1,CRITICAL,CVSS:3.1/AV:N/AC:L/PR:H/UI:N/S:C/C:H/I:H/A:H,"[CWE-266, CWE-269]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
7,threatcluster_0000007,<NA>,A vulnerability was found in Tenda HG10 300001...,2026-09-06,9.8,CRITICAL,CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H,"[CWE-119, CWE-120]",None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
8,threatcluster_0000008,<NA>,The Real Estate Papi WordPress theme through 1...,2026-09-06,None,None,None,None,None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>
9,threatcluster_0000009,<NA>,The HivePress Authentication plugin for WordPr...,2026-09-06,7.5,HIGH,CVSS:3.1/AV:N/AC:H/PR:N/UI:R/S:U/C:H/I:H/A:H,[CWE-287],None,None,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,<NA>


## 9. Data-quality checks

In [9]:
quality = {
    "total_rows": len(combined),
    "unique_uid": combined["uid"].nunique(),
    "rows_missing_cve_id": int(combined["cve_id"].isna().sum()),
    "duplicate_uid_count": int(combined["uid"].duplicated().sum()),
    "duplicate_source_cve_pairs": int(combined.duplicated(["source", "cve_id"]).sum()),
    "blank_threat_category": int(combined["threat_category"].isna().sum()),
}
display(pd.Series(quality, name="value").to_frame())

# Show duplicates within a source so they can be investigated, not silently dropped.
duplicate_pairs = combined[
    combined["cve_id"].notna() & combined.duplicated(["source", "cve_id"], keep=False)
].sort_values(["source", "cve_id"])
print("Rows involved in duplicate (source, cve_id) pairs:", len(duplicate_pairs))
display(duplicate_pairs.head(30))

,value
total_rows,61965
unique_uid,61965
rows_missing_cve_id,61965
duplicate_uid_count,0
duplicate_source_cve_pairs,61963
blank_threat_category,61965


Rows involved in duplicate (source, cve_id) pairs: 0


,uid,cve_id,description,published_date,cvss_v3_score,cvss_v3_severity,cvss_v3_vector,cwe_ids,affected_vendors,affected_products,...,techniques,techniques_derived,label_sources,attack_version,cvss_vector,cvss_version,cwes,cpes,cwes_predicted,threat_category


## 10. Export final dataset and reports

The output files are saved inside `cti_dataset_output/` in the notebook's current working directory.

In [10]:
csv_path = OUTPUT_DIR / "SMAH-CTI67.csv"
parquet_path = OUTPUT_DIR / "SMAH-CTI67.parquet"
missing_path = OUTPUT_DIR / "missing_values_report.csv"
schema_path = OUTPUT_DIR / "dataset_schema_report.csv"
overlap_path = OUTPUT_DIR / "overlapping_cve_ids.csv"

combined.to_csv(csv_path, index=False, encoding="utf-8-sig")
combined.to_parquet(parquet_path, index=False)

missing_report = pd.DataFrame({
    "column": combined.columns,
    "dtype": [str(combined[c].dtype) for c in combined.columns],
    "missing_count": [int(combined[c].isna().sum()) for c in combined.columns],
    "missing_pct": [round(float(combined[c].isna().mean() * 100), 2) for c in combined.columns],
    "non_null_count": [int(combined[c].notna().sum()) for c in combined.columns],
})
missing_report.to_csv(missing_path, index=False)

schema_report = pd.DataFrame({
    "column": combined.columns,
    "dtype": [str(combined[c].dtype) for c in combined.columns],
    "source_threatcluster_non_null": [
        int(combined.loc[combined["source"] == "threatcluster", c].notna().sum())
        if c in combined.columns else 0 for c in combined.columns
    ],
    "source_circl_non_null": [
        int(combined.loc[combined["source"] == "circl", c].notna().sum())
        if c in combined.columns else 0 for c in combined.columns
    ],
})
schema_report.to_csv(schema_path, index=False)

if len(sources_present) >= 2:
    pd.DataFrame({"cve_id": sorted(overlap)}).to_csv(overlap_path, index=False)

print("Export complete:")
for p in [csv_path, parquet_path, missing_path, schema_path]:
    print(f"- {p.resolve()} | {p.stat().st_size:,} bytes")
if len(sources_present) >= 2:
    print(f"- {overlap_path.resolve()} | {overlap_path.stat().st_size:,} bytes")

Export complete:
- D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new\cti_dataset_output\SMAH-CTI67.csv | 42,005,277 bytes
- D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new\cti_dataset_output\SMAH-CTI67.parquet | 14,672,024 bytes
- D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new\cti_dataset_output\missing_values_report.csv | 1,202 bytes
- D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new\cti_dataset_output\dataset_schema_report.csv | 945 bytes
- D:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new\cti_dataset_output\overlapping_cve_ids.csv | 8 bytes


## 11. Interpretation notes for thesis use

- **Authentication is not a missing-value imputation method.** Compare source values with exported values to find where missingness originates.
- Keep missing values missing unless a documented, defensible imputation or enrichment method is applied.
- `uid` identifies a row; `cve_id` links records referring to the same vulnerability.
- Keep `published_date` separate from `kev_added_date` and `kev_due_date`.
- Do not treat an absent technique label as a negative label.
- If your target label is created from `in_kev`, `has_exploit`, `exploit_count`, `ransomware_use`, or keywords, do not use those same signals as model inputs for that target (target leakage).
- For temporal emerging-threat evaluation, use a time-based split; for overlapping CVEs across sources, group by `cve_id` when splitting to reduce leakage.
- Document any rule-based labels and manually validate a sample before reporting classification metrics.

In [3]:
import pandas as pd
from pathlib import Path

# Auto-detect project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
print('CWD         :', Path.cwd())
print('PROJECT_ROOT:', PROJECT_ROOT)

# Try both likely filenames
candidates = [
    PROJECT_ROOT / 'dataset' / 'SMAH-CTI67.csv',
    PROJECT_ROOT / 'dataset' / 'cti_dataset_clean.csv',
    PROJECT_ROOT / 'dataset' / 'cti_dataset.csv',
]
for p in candidates:
    print(f'  {p.name:30s} exists={p.exists()}')

# Load whichever exists
CLEAN = next((p for p in candidates if p.exists()), None)
if CLEAN is None:
    print('\n❌ None found. Listing dataset/ folder:')
    if (PROJECT_ROOT / 'dataset').exists():
        for f in sorted((PROJECT_ROOT / 'dataset').iterdir()):
            print('  ', f.name, f'({f.stat().st_size:,} bytes)')
    raise FileNotFoundError('no dataset CSV found')

print(f'\n✅ Using: {CLEAN}')
df = pd.read_csv(CLEAN)
print(f'Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols')
print('\nColumns:', df.columns.tolist())
print('\nFirst 3 rows:')
print(df.head(3).to_string())

CWD         : d:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new
PROJECT_ROOT: d:\Projects\done\thesis\cyber_threat_classification_n_emerging_threat_detection\new
  SMAH-CTI67.csv                 exists=False
  cti_dataset_clean.csv          exists=False
  cti_dataset.csv                exists=False

❌ None found. Listing dataset/ folder:


FileNotFoundError: no dataset CSV found